# 1. Migracao default -> raw

Copia dados das tabelas em `default` para as tabelas correspondentes em `raw` (convencao: raw.{schema}_{table}).

# 2. Configuracao e conexao ClickHouse

In [192]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _root = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd()
    for p in [Path.cwd(), _root, Path.cwd().parent]:
        env_path = p / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            break
except Exception:
    pass

CH_HOST = os.getenv("CLICKHOUSE_HOST", "")
CH_PORT = int(os.getenv("CLICKHOUSE_PORT", "8443"))
CH_USER = os.getenv("CLICKHOUSE_USER", "default")
CH_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CH_DEFAULT_DB = os.getenv("CLICKHOUSE_DATABASE", "default")
CH_SECURE = os.getenv("CLICKHOUSE_SECURE", "true").lower() == "true"

In [193]:
import clickhouse_connect

ch_client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
    database=CH_DEFAULT_DB,
    secure=CH_SECURE,
    connect_timeout=60,
    send_receive_timeout=300,
)
ch_client.command("SELECT 1")

1

# 3. Mapeamento default -> raw

default.{table} -> raw.{schema}_{table} (conforme tables.yaml).

In [194]:
DEFAULT_TO_RAW = [
    ("depara_cliente", "ginf_depara_cliente"),
    ("base_cep_completa", "ginf_base_cep_completa"),
    ("tst_contratos_bi", "ginf_tst_contratos_bi"),
    ("sc5030", "siga_sc5030"),
    ("sc6030", "siga_sc6030"),
    ("tst_historico_solicitacoes", "ginf_tst_historico_solicitacoes"),
    ("tst_solicit_cadastradas", "ginf_tst_solicit_cadastradas"),
    ("sd2030", "siga_sd2030"),
    ("sf2030", "siga_sf2030"),
    ("ztx030", "siga_ztx030"),
    ("tst_contratos", "ginf_tst_contratos"),
]

# 4. Criar database raw e listar tabelas existentes em default

In [195]:
ch_client.command("CREATE DATABASE IF NOT EXISTS raw")

existing_default = set(
    row["name"]
    for row in ch_client.query(
        "SELECT name FROM system.tables WHERE database = %(db)s",
        parameters={"db": CH_DEFAULT_DB},
    ).named_results()
)
print(f"Tabelas em {CH_DEFAULT_DB}:", sorted(existing_default))

Tabelas em default: ['SC_USER', 'base_cep_completa', 'bistage', 'depara_cliente', 'migration_progress', 'sc5030', 'sc6030', 'sd2030', 'sf2030', 'tracker_raw', 'tst_contratos', 'tst_historico_solicitacoes', 'tst_solicit_cadastradas', 'ztx030']


# 5. Copiar cada tabela: default -> raw

In [196]:
def table_exists(client, database, table):
    r = client.query(
        "SELECT 1 FROM system.tables WHERE database = %(db)s AND name = %(name)s LIMIT 1",
        parameters={"db": database, "name": table},
    ).result_rows
    return bool(r)


def migrate_table(client, default_table, raw_table):
    if not table_exists(client, CH_DEFAULT_DB, default_table):
        return "skip", 0, f"{CH_DEFAULT_DB}.{default_table} nao existe"
    client.command(f"DROP TABLE IF EXISTS raw.{raw_table}")
    client.command(f"CREATE TABLE raw.{raw_table} AS {CH_DEFAULT_DB}.{default_table}")
    n = client.query(f"SELECT count() FROM {CH_DEFAULT_DB}.{default_table}").first_row[0]
    if n > 0:
        client.command(f"INSERT INTO raw.{raw_table} SELECT * FROM {CH_DEFAULT_DB}.{default_table}")
    return "ok", n, f"{n} linhas"

In [197]:
results = []
for default_tbl, raw_tbl in DEFAULT_TO_RAW:
    status, count, msg = migrate_table(ch_client, default_tbl, raw_tbl)
    results.append((default_tbl, raw_tbl, status, count, msg))
    print(f"{CH_DEFAULT_DB}.{default_tbl} -> raw.{raw_tbl}: {status} ({msg})")

default.depara_cliente -> raw.ginf_depara_cliente: ok (93 linhas)
default.base_cep_completa -> raw.ginf_base_cep_completa: ok (100030 linhas)
default.tst_contratos_bi -> raw.ginf_tst_contratos_bi: skip (default.tst_contratos_bi nao existe)
default.sc5030 -> raw.siga_sc5030: ok (50012 linhas)
default.sc6030 -> raw.siga_sc6030: ok (50011 linhas)
default.tst_historico_solicitacoes -> raw.ginf_tst_historico_solicitacoes: ok (28780000 linhas)
default.tst_solicit_cadastradas -> raw.ginf_tst_solicit_cadastradas: ok (150688 linhas)
default.sd2030 -> raw.siga_sd2030: ok (50012 linhas)
default.sf2030 -> raw.siga_sf2030: ok (469906 linhas)
default.ztx030 -> raw.siga_ztx030: ok (50053 linhas)
default.tst_contratos -> raw.ginf_tst_contratos: ok (6107641 linhas)


# 6. Resumo

In [198]:
import pandas as pd

df = pd.DataFrame(
    results,
    columns=["default_table", "raw_table", "status", "rows", "message"],
)
df

,default_table,raw_table,status,rows,message
0,depara_cliente,ginf_depara_cliente,ok,93,93 linhas
1,base_cep_completa,ginf_base_cep_completa,ok,100030,100030 linhas
2,tst_contratos_bi,ginf_tst_contratos_bi,skip,0,default.tst_contratos_bi nao existe
3,sc5030,siga_sc5030,ok,50012,50012 linhas
4,sc6030,siga_sc6030,ok,50011,50011 linhas
5,tst_historico_solicitacoes,ginf_tst_historico_solicitacoes,ok,28780000,28780000 linhas
6,tst_solicit_cadastradas,ginf_tst_solicit_cadastradas,ok,150688,150688 linhas
7,sd2030,siga_sd2030,ok,50012,50012 linhas
8,sf2030,siga_sf2030,ok,469906,469906 linhas
9,ztx030,siga_ztx030,ok,50053,50053 linhas


# 7. Remover da default as tabelas que foram copiadas para raw

So serao dropadas tabelas com status ok (copiadas com sucesso).

In [199]:
for default_tbl, raw_tbl, status, count, msg in results:
    if status != "ok":
        continue
    ch_client.command(f"DROP TABLE IF EXISTS {CH_DEFAULT_DB}.{default_tbl}")
    print(f"DROP {CH_DEFAULT_DB}.{default_tbl}")

DROP default.depara_cliente
DROP default.base_cep_completa
DROP default.sc5030
DROP default.sc6030
DROP default.tst_historico_solicitacoes
DROP default.tst_solicit_cadastradas
DROP default.sd2030
DROP default.sf2030
DROP default.ztx030
DROP default.tst_contratos


# 8. Sanitizacao: bancos sem dados

Lista databases, identifica os que nao tem dados (sem tabelas ou total_rows=0) e remove-os. Bancos protegidos: default, raw, trusted, gold.

In [210]:
PROTECTED_DATABASES = {"default", "raw", "trusted", "gold", "system", "INFORMATION_SCHEMA", "information_schema"}

rows = ch_client.query("""
    SELECT database, sum(total_rows) AS total_rows
    FROM system.tables
    WHERE database NOT IN ('system', 'INFORMATION_SCHEMA', 'information_schema')
    GROUP BY database
""").named_results()

all_dbs = set(
    r["name"] for r in ch_client.query("SELECT name FROM system.databases").named_results()
)
dbs_with_data = {r["database"] for r in rows if (r.get("total_rows") or 0) > 0}
dbs_with_tables_no_data = {r["database"] for r in rows if (r.get("total_rows") or 0) == 0}
dbs_no_tables = all_dbs - {r["database"] for r in rows} - PROTECTED_DATABASES
empty_to_drop = (dbs_with_tables_no_data | dbs_no_tables) - PROTECTED_DATABASES

print("Databases com dados:", sorted(dbs_with_data))
print("Databases vazios (sem tabelas ou total_rows=0):", sorted(empty_to_drop))

Databases com dados: ['default', 'ginf', 'raw', 'scot', 'siga', 'track_bronze', 'track_gold', 'track_silver', 'tracker_raw', 'uk']
Databases vazios (sem tabelas ou total_rows=0): ['airbyte', 'bronze', 'ginf', 'scot', 'siga', 'silver', 'stage', 'track_bronze', 'track_gold', 'track_silver', 'tracker_bronze', 'tracker_gold', 'tracker_raw', 'tracker_refined', 'tracker_silver', 'tracker_trusted', 'uk']


In [211]:
for db in sorted(empty_to_drop):
    ch_client.command(f"DROP DATABASE IF EXISTS {db}")
    print(f"DROP DATABASE {db}")

DROP DATABASE airbyte
DROP DATABASE bronze
DROP DATABASE ginf
DROP DATABASE scot
DROP DATABASE siga
DROP DATABASE silver
DROP DATABASE stage
DROP DATABASE track_bronze
DROP DATABASE track_gold
DROP DATABASE track_silver


2026-02-08 23:29:23 | WARNING  | clickhouse_connect.driver.httpclient:_raw_request | Unexpected Http Driver Exception
2026-02-08 23:29:23 | INFO     | py4j.clientserver:close | Closing down clientserver connection
2026-02-08 23:29:23 | INFO     | py4j.clientserver:close | Closing down clientserver connection
2026-02-08 23:29:23 | INFO     | py4j.clientserver:close | Closing down clientserver connection
2026-02-08 23:29:23 | INFO     | py4j.clientserver:close | Closing down clientserver connection


OperationalError: Error HTTPSConnectionPool(host='e1a1lieug8.us-central1.gcp.clickhouse.cloud', port=8443): Read timed out. (read timeout=300) executing HTTP request attempt 1 https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443

# 9. Estado final: organizacao (default, raw, trusted, gold)

In [ ]:
for db in ["default", "raw", "trusted", "gold"]:
    try:
        tbls = ch_client.query(
            "SELECT name, total_rows FROM system.tables WHERE database = %(db)s ORDER BY name",
            parameters={"db": db},
        ).named_results()
        total = sum(t.get("total_rows") or 0 for t in tbls)
        print(f"{db}: {len(tbls)} tabelas, {total} linhas")
        for t in tbls[:20]:
            print(f"  - {t['name']}: {t.get('total_rows') or 0}")
        if len(tbls) > 20:
            print(f"  ... e mais {len(tbls) - 20} tabelas")
    except Exception as e:
        print(f"{db}: (nao existe ou erro) {e}")